# **LM Embeddings Workshop**
**By Idan Asher Blank ([email](mailto:iblank@psych.ucla.edu)), Summer 2026**
<br>
<br>
This notebook has the following components:
1. **Guided demo (sections 1 & 2):** pull contextual embeddings out of BERT and see how the same word gets different representations depending on its sentence context.
3. **Embedding similarities exercise (section 3):** use BERT embeddings to look at how this Transformer represents different instances of an English syntactic alternation (double-object vs. prepositional-object datives).d
5. **Probing exercise (sections 4 & 5):** train a simple probing classifier to test whether contextual embeddings contain implicit information about sentence grammaticality.

**Run each block in order.** If you edit something and the results look wrong, use `Runtime -> Run all`, or re-run every block from your edit onward.

## **Preliminaries: install libraries**
We need one library that isn't already in Colab: `minicons`, a toolkit built by Dr. Kanishka Misra on top of the Hugging Face `transformers` library. This package makes it easy to pull a single token's contextual embedding out of a language model.

**Note:** The install text that appears below (often in orange or red) is normal Python installation chatter, not an error&mdash;as long as there's no line that says `ERROR`, you're fine.

In [ ]:
# The -q flag just quiets down the installer's normal chatter.
!pip install minicons -q
from minicons import cwe
# CWE = "Contextual Word Embeddings". This object wraps a LM and knows how
# to find a specific token's vector inside a sentence for us.

import os
import numpy as np    # numpy: a library for working with grids of numbers (i.e., arrays)
import pandas as pd   # pandas: a library for working with data tables (data frames)

# Some Hugging Face downloads try to check for a login "token" (a saved
# password-like credential). None of the models/data in this workshop are
# private, so we turn that check off to avoid an unnecessary "grant access
# to secret" popup in Colab. However, if you have used both Colab and Hugging Face
# before and you have a secret key called "HF_TOKEN", you can allow this notebook
# access to that key, and the loading of models may be quicker.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# We use the CPU (regular processor) everywhere in this notebook, on purpose -
# no need to go hunting for a GPU runtime. Everything here is small enough to
# run comfortably on CPU. But when you do this at home, you can use Colab's
# free GPU (T4). To do so, from the Runtime menu on top, click "change runtime type"
# and choose "T4 GPU". Then re-run the notebook from the top.
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Setup complete. Using device:", device)

## **Section 1: one word, several meanings**
### 1.1: Load BERT and inspect its architecture

We're loading `bert-base-uncased`, a widely-used Transformer in psycholinguistic research. It is a **bidirectional** transformer, meaning that the representation of each token is influenced both the preceding tokens and following tokens: a token can attend to past *and* future tokens.

This part may take a minute the first time, because we're downloading the LM (i.e., all the parameters&mdash;the static embeddings of all the tokens, the Query, Key, and Value matrices of every attention head in every layer, all the weights of the feed-forward networks, etc.).

Before we start pulling out contextual embeddings, we'll also see what `bert-base-uncased` actually looks like on the inside: how many processing stages it has, how long each token's embedding is, and roughly how many parameters it contains (i.e., adjustable numbers&mdash;the values the model learned during training). The package `minicons` uses a model object from Hugging Face called `model.model`; its `.config` field holds these architecture settings.

In [ ]:
# You can delete device=device below; in that case, minicons will just use cpu
model = cwe.CWE("bert-base-uncased", device=device)
print("BERT loaded and ready.")

bert_config = model.model.config

n_layers = bert_config.num_hidden_layers           # number of transformer blocks stacked on top of each other
hidden_size = bert_config.hidden_size              # length of the embedding representing each word
n_heads = bert_config.num_attention_heads          # how many "attention heads" in each block
vocab_size = bert_config.vocab_size                # how many distinct tokens are in BERT's vocabulary
n_params = sum(p.numel() for p in model.model.parameters())  # total count of the model's learned numbers

print(f"Transformer blocks (\"layers\"):          {n_layers}")
print(f"Embedding size per token (hidden size): {hidden_size}")
print(f"Attention heads per block:              {n_heads}")
print(f"Vocabulary size:                        {vocab_size:,} word-pieces")
print(f"Total parameters:                       {n_params:,} (approximately {n_params / 1e6:.0f} million)")

### 1.2: One sentence, one target word

Here's our first sentence, with the target word in **bold**:

> She hit the ball with the **bat** and it flew over the fence.

`minicons` wants its input as a list of `(sentence, word)` pairs. **The word must appear in the sentence exactly as spelled** (same capitalization, same punctuation attached or not) or extraction will raise an error.

In [ ]:
# A list containing one (sentence, target_word) tuple ("tuple" = a small,
# fixed-size grouping of values, here a sentence paired with the word we want an embedding for).

instance = [("She hit the ball with the bat and it flew over the fence.", "bat")]
print(instance)

### 1.3: Extract the contextual embedding and check its shape

Now we ask BERT for the contextual embedding for the token "bat" (specified above), at a certain layer, defined with the `LAYER` variable below (out of 12 hidden layers).

This contextual embedding is at the end of the block (hidden layer)&mdash;after attention, the feed-forward multi-layer perceptron, and any layer normalization.

The output (`reps`) has a `.shape`&mdash;checking the shape is a cheap sanity check that we got what we expected.

We'll also print the first few values of the contextual embeddings, for good measure (these are not interpretable on their own).

In [ ]:
# The LAYER variable sets the layer from which we extract the contextual embedding in the the residual stream

LAYER = 8;

reps = model.extract_representation(instance, layer=LAYER)

print(f"Shape: {reps.shape} -- that's {reps.shape[0]} token, turned into a {reps.shape[1]}-number embedding")
print()
print(reps[0,0:10])

### 1.4: The demo&mdash;three sentences, two senses

Now the real test: three sentences, all containing **"bat"**, but only two of them mean the same thing. Sentences 1 and 2 use the *tool for hitting* meaning; sentence 3 uses the *flying mammal* meaning.

1. "She hit the ball with the **bat** and it flew over the fence." (tool for hitting)
2. "For his birthday the young boy got a wooden **bat**." (tool for hitting)
3. "The **bat** flew out of the cave and found some leftover cake on the ground." (flying mammal)

If BERT's contextual embeddings are sensitive to meaning, the embeddings of "bat" in sentences 1 and 2 should be more similar to each other than either is to the contextual embedding of "bat" in sentence 3.

To make this test more challenging, note that both 1 and 2 share some similarities with 3 that are unrelated to **bat**:
* Sentences 1 & 3 both have the verb "flew" (but in the first sentence, the "it" that flew refers to the ball, not the bat)
* Sentences 2 has the word "birthday", which is semantically related to the word "cake" in sentence 3

In [ ]:
# Define the sentences
bat_sentences = [
    ("She hit the ball with the bat and it flew over the fence.", "bat"),
    ("For his birthday the young boy got a wooden bat.", "bat"),
    ("The bat flew out of the cave and found some leftover cake on the ground.", "bat")
]

# Extract the contextual embeddings of "bat" and verify their size
reps_bat = model.extract_representation(bat_sentences, layer=LAYER)

print(f"Extracting embeddings from layer {LAYER}")
print(f"Shape: {reps_bat.shape} -- that's {reps_bat.shape[0]} tokens, each turned into a {reps_bat.shape[1]}-number embedding")

### 1.5: Look at how BERT tokenizes each sentence

Let's observe how BERT takes the words in our sentences and converts them to tokens from its own vocabulary. `model.tokenizer` is the tokenizer BERT was trained with; `.tokenize()` shows exactly which pieces a sentence gets split into, without running the Transformer itself.

This is worth checking before extracting a specific word's embedding&mdash;if a word gets split into multiple pieces, ``minicons`` has to know which piece(s) to grab (luckily, it figures this out automatically).

In [ ]:
for sentence, word in bat_sentences:
    print(sentence)
    print(f"  -> tokens: {model.tokenizer.tokenize(sentence)}")
    print()

# You can uncomment the line below to see the tokenization of the word "psycholinguistics"
# print(f"  -> tokens: {model.tokenizer.tokenize([("psycholinguistics")])}")

**What to notice:** A `##` prefix marks a token that "goes together" with the previous token rather than starting a new word. In our sentences, there is one word that was tokenized into two tokens.

Some words may be tokenized into even more tokens: for example, you an uncomment a command in the code above to tokenize "psycholinguistics".

Tokens that are word initial vs. tokens that are continuations have distinct static embeddings in the model (e.g., "over" in sentence 1 has a different static embedding than "##over" in sentence 3). If you are interested in some numbers: of the ~24,700 word-initial tokens, about 11% have a matching continuation-form token (the other tokens only exist as word-initial).

#### What happens when a target word splits into multiple tokens?
`"bat"` is a single whole token in all three sentences, but plenty of words get "broken up" into multiple sub-word tokens. When you ask for a contextual embedding from such a word, `minicons` doesn't just grab one token and ignore the rest. It finds every token belonging to that word inside the tokenized sentence, and then **averages their embeddings together** into a single embedding. This way, you still end up with one embedding, which has the same length as the embeddings of single-token words. This all happens automatically inside `extract_representation()`; nothing in this notebook's own code has to handle it.

### 1.6: Compute similarities

We'll compare the 3 embeddings of "bat" using **cosine similarity**: a number between -1 and 1 that measures how similar two embeddings are in terms of their *directions*, ignoring their length (this is different from, say, Euclidean distance). A cosine similarity of 1 means identical direction, 0 means unrelated, -1 means opposite directions. It's the standard way to compare two embeddings.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

reps_bat_np = reps_bat.numpy() # cosine_similarity expects a NumPy array, so we convert the tensor first.
sim_matrix = cosine_similarity(reps_bat_np)

sim_12 = sim_matrix[0, 1]
sim_13 = sim_matrix[0, 2]
sim_23 = sim_matrix[1, 2]

print()
print(f"Similarity, Sentences 1 & 2 (tool & tool): {sim_12:.3f}")
print(f"Similarity, Sentences 1 & 3 (tool & animal):  {sim_13:.3f}")
print(f"Similarity, Sentences 2 & 3 (tool & animal):  {sim_23:.3f}")

**What to notice:** compare the three numbers above&mdash;the tool/tool pair should stand out as clearly higher than either tool/animal pair, even though all three sentences use the word "bat."

### Dealing with anisotropy
How are contextual embeddings organized in their high-dimensional representational space? It turns out that they are not "evenly" spread across all dimensions (or hidden units). As described in [this paper](https://aclanthology.org/2021.emnlp-main.372/), a small number of hidden units "dominate" the organization of these embeddings, because the activations of these units show very high variance (across different inputs). This property of high-dimensional spaces is called **anisotropy**. It turns out that these dimensions do not have a special role in the LM's behavior: they do not code "more important" information compared to other units.

Because the "rogue" dimensions have a disproportionate influence on the geometry of the representational space, they distort measures of similarity between embeddings. In effect, cosine simialrity would reflect mostly information from these few dimensions; information coded by other dimensions&mdash;which can be linguistically relevant&mdash;would not have a strong influence on which embeddings are more vs. less similar to one another.

What can we do? One solution is to normalize, or z-score, the activation of each hidden unit (i.e., z-score the embedding along each dimension): instead of taking a unit's raw value, we subtract its average activation (measured across many, many different inputs) and divide by the standard deviation of its activations. The next section will show you how to do this.





### 1.7: Load normalization statistics, define `normalize()`
For z-scoring each of BERT's 768 units (in a given layer), we will use statistics from a large external corpus (a section of [COCA](https://www.english-corpora.org/coca/?__cf_chl_rt_tk=pYJGmaM2iTEKCmcZvPgnp3UvP86d9O4q9EgQSAQIstI-1784844053-1.0.1.1-CVajYB3cNvc9bpr0bYxgXq3cf_hvstdc6JXiMLg7eKI)). The next section loads those statistics and defines a normalization function.




In [ ]:
BERT_MEANS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/bert_unit_means.tsv"
BERT_SDS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/bert_unit_sds.tsv"

# Wide format, no header row: row i holds the 768 per-unit values for layer i.
bert_means = pd.read_csv(BERT_MEANS_URL, sep="\t", header=None)
bert_sds = pd.read_csv(BERT_SDS_URL, sep="\t", header=None)

print(f"Means table shape: {bert_means.shape} -- {bert_means.shape[0]} layers x {bert_means.shape[1]} units")
print(f"SDs table shape:   {bert_sds.shape} -- {bert_sds.shape[0]} layers x {bert_sds.shape[1]} units")

# A function that takes contextual embeddings and normalizes the value of each hidden unit relative
# to the means and SDs in the variables above
def normalize(embeddings, layer, means_df, sds_df):
    """Z-score each of the 768 dimensions using external corpus statistics
    for this layer. means_df/sds_df are WIDE-format tables loaded with header=None:
    row i holds per-unit values for layer i (row 0 = static embeddings;
    rows 1-12 = after each transformer block). Pull out the row
    for the requested layer, then apply it column-wise."""
    layer_means = means_df.iloc[layer].values
    layer_sds = sds_df.iloc[layer].values
    return (embeddings - layer_means) / layer_sds

**Note:** these matrices have values for 13 layers&mdash;these include the static embedding layer + 12 hidden layers.

### 1.8: Recompute the three similarities, normalized

Same three "bat" sentences, same layer from sections 1.3-1.6, but now activations are normalized before comparing the contextual embeddings (using the function we just defined).

In [ ]:
reps_bat_norm = normalize(reps_bat_np, layer=LAYER, means_df=bert_means, sds_df=bert_sds)
sim_matrix_norm = cosine_similarity(reps_bat_norm)

sim_12_norm = sim_matrix_norm[0, 1]
sim_13_norm = sim_matrix_norm[0, 2]
sim_23_norm = sim_matrix_norm[1, 2]

print("RAW (section 1.6):")
print(f"  Sentences 1 & 2 (tool & tool): {sim_12:.3f}")
print(f"  Sentences 1 & 3 (tool & animal):  {sim_13:.3f}")
print(f"  Sentences 2 & 3 (tool & animal):  {sim_23:.3f}")
print()

print("NORMALIZED (this section):")
print(f"  Sentences 1 & 2 (tool & tool): {sim_12_norm:.3f}")
print(f"  Sentences 1 & 3 (tool & animal):  {sim_13_norm:.3f}")
print(f"  Sentences 2 & 3 (tool & animal):  {sim_23_norm:.3f}")

**What to notice:** normalizing should widen the gap between the tool/tool pair and the tool/animal pairs relative to the raw numbers&mdash;once we correct for anisotropy, the meaningful similarity differences are easier to see (even though, in this case, the *ranking* of the three pairs doesn't change.)

In [ ]:
#@title 1.9: Question: Which pair of sentences has the HIGHEST cosine similarity? (Choose an answer and press play)
your_answer = "Sentences 1 & 2" #@param ["Sentences 1 & 2", "Sentences 1 & 3", "Sentences 2 & 3"]

# Live-graded: thie code recomputes the actual highest-similarity pair from your own
# results above, rather than hardcoding an answer, so this stays correct
# even if your numbers differ slightly from someone else's.
pair_names = ["Sentences 1 & 2", "Sentences 1 & 3", "Sentences 2 & 3"]
pair_values = [sim_12_norm, sim_13_norm, sim_23_norm]
correct_pair = pair_names[int(np.argmax(pair_values))]

if your_answer == correct_pair:
    print(f"✅ Correct! {correct_pair} had the highest similarity (fter normalizing activations, the value is {max(pair_values):.3f}).")
else:
    print(f"❌ Not quite. The highest similarity was actually {correct_pair} (after normalizing activations, the value is {max(pair_values):.3f}).")

### 1.10: Changing the layer

BERT has 12 hidden layers of representation (i.e., 12 Transformer blocks). Let's recompute the sentence similarities, but for (normalized) embeddings from a different layer (instead of layer 8). Below, choose a number between 1 and 12 for `NEW_LAYER`, and check whether that layer is as "good" as layer 8 at telling the two meanings of "bat" apart.



In [ ]:
# EDIT ME: ADD LAYER NUMBER AFTER THE "=" SIGN, BETWEEN 1 AND 12
NEW_LAYER = 5

# Extract representation
reps_bat_layerN = model.extract_representation(bat_sentences, layer=NEW_LAYER)

# Normalize embeddings, then calculate similarities
reps_bat_layerN_np = reps_bat_layerN.numpy()
reps_bat_layerN_norm = normalize(reps_bat_layerN_np, layer=NEW_LAYER, means_df=bert_means, sds_df=bert_sds)
sim_matrix_layerN_norm = cosine_similarity(reps_bat_layerN_norm)

sim_12_lN = sim_matrix_layerN_norm[0, 1]
sim_13_lN = sim_matrix_layerN_norm[0, 2]
sim_23_lN = sim_matrix_layerN_norm[1, 2]

# Print the results
print(f"Layer {NEW_LAYER}:")
print(f"  Sentences 1 & 2 (tool & tool): {sim_12_lN:.3f}")
print(f"  Sentences 1 & 3 (tool & animal):  {sim_13_lN:.3f}")
print(f"  Sentences 2 & 3 (tool & animal):  {sim_23_lN:.3f}")

### 1.11: Comparing all the hidden layers

The code below will run the same calculation as the previous one, but for each hidden layer in turn, and plot the results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract contextual embeddings from all hidden layers
reps_all_layers = model.extract_representation(bat_sentences, layer="all") # One 768-dim "bat" embedding per sentence, for every layer, extracted all at once
n_layers = len(reps_all_layers)

# Compute similarities for "bat" across the 3 sentences, for each layer separately
sims_12, sims_13, sims_23 = [], [], []
for curr_layer in range(n_layers):
    layer_vecs = reps_all_layers[curr_layer].numpy()  # shape is (3, 768): rows are sentence 1, 2, 3
    layer_vecs_norm = normalize(layer_vecs, layer=curr_layer, means_df=bert_means, sds_df=bert_sds)  # Z-score each hidden unit
    sim_matrix = cosine_similarity(layer_vecs_norm)
    sims_12.append(sim_matrix[0, 1])
    sims_13.append(sim_matrix[0, 2])
    sims_23.append(sim_matrix[1, 2])

sims_12 = sims_12[1:]   # we only keep layers 1-12; we'll deal with the static embedding layer later
sims_13 = sims_13[1:]   # same
sims_23 = sims_23[1:]   # same

# Plot the similarities
layers = np.arange(len(sims_12))+1
bar_width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(layers - bar_width, sims_12, width=bar_width, label="Sentences 1 & 2 (tool & tool)", color="tab:blue")
ax.bar(layers, sims_13, width=bar_width, label="Sentences 1 & 3 (tool & animal)", color="dimgray")
ax.bar(layers + bar_width, sims_23, width=bar_width, label="Sentences 2 & 3 (tool & animal)", color="lightgray")

ax.set_xlabel("Layer")
ax.set_ylabel("Cosine similarity")
ax.set_title('Pairwise similarity of "bat" across all 12 hidden layers')
ax.set_xticks(layers)
ax.legend()
plt.tight_layout()
plt.show()

**What to notice**: watch whether the "tool & tool" bar (blue) pulls away from the other two bars as layer index increases. If the representation of contextual meaning sharpens with depth, blue should separate from gray/lightgray more in later layers than in early ones. At the same time, remember that the last layer comes right before the next-token prediction stage, so its representations are often somewhat more influenced by this goal than representations in the "middle" of the transformer.

Overall, many studies of Transformers find that semantic information is captured well by middle-late layers (like layer 8 that we originally inspected).

### 1.12: The static-embedding layer

Finally, let's check the static embeddings of "bat" in the 3 sentences. We refer to this layer as layer 0.

We cannot extract these embeddings by using `NEW_LAYER = 0` in the code from 1.11, because the represenation of layer 0 there includes a combination of the static token embeddings *and* the positional embedding (the latter encodes the position of each token in the sentence). Instead, we set the layer to `"static"`. The code below carries out this calculation.

In [ ]:
# Extract representations from the "static" layer
reps_bat_static = model.extract_representation(bat_sentences, layer="static")

# Normalize the embeddings and calculate cosine similarities
reps_bat_static_np = reps_bat_static.numpy()
reps_bat_static_norm = normalize(reps_bat_static_np, layer=0, means_df=bert_means, sds_df=bert_sds)
sim_matrix_static_norm = cosine_similarity(reps_bat_static_norm)

sim_12_static = sim_matrix_static_norm[0, 1]
sim_13_static = sim_matrix_static_norm[0, 2]
sim_23_static = sim_matrix_static_norm[1, 2]

# Print the results
print("Static embedding layer:")
print(f"  Sentences 1 & 2 (tool & tool): {sim_12_static:.3f}")
print(f"  Sentences 1 & 3 (tool & animal):  {sim_13_static:.3f}")
print(f"  Sentences 2 & 3 (tool & animal):  {sim_23_static:.3f}")

**What to notice**: What do these similarity values mean? And why did we get these values?

Note that, if you change `"static"` to `0`, you will get different similarities. The reason is that, for `"static"`, minicons extracts the static embeddings themselves. But for `layer=0`, minicons extracts an embedding that also incorporates positional information, and which has undergone layer normalization (not the kind of normalization we do here, but the kind described towards the end of my LM video).
<br>
<br>
<br>

## **Section 2: Try your own sentences!**

### 2.1: Write sentences

Now try the same exercise, but on new stimuli. Write your own sentences that contain the **same word**, used in different contexts. You can use homonyms (like "bat"), a polysemous word with different senses (like "run"), a word whose construal varies depending on the syntactic structure its in, or a word where different aspects of meaning may be emphasized in different contexts.

Edit the list below: replace the placeholder sentences with your own. Keep the `(sentence, word)` format, and remember the word must appear in the sentence exactly as spelled. If you want more then 3 sentences, just add more rows with the same structure.

Don't forget to choose which layer you want to extract embeddings from.

In [ ]:
# EDIT ME: REPLACE THESE PLACEHOLDERS WITH YOUR OWN SENTENCES
my_sentences = [
    ("The plant on my desk needs more sunlight.", "plant"),
    ("This factory is a plant that makes car parts.", "plant"),
    ("I need to water the plant before it wilts.", "plant"),
]

# EDIT ME: CHOOSE A HIDDEN LAYER (1-12)
MY_LAYER = 8

# Print the sentences
for sentence in my_sentences: print(sentence)

### 2.2: Extract and compare

The code below extracts contextual embeddings for the relevant token from all of your sentences at once, and prints the full pairwise similarity matrix as a small table.

In [ ]:
# Extract contextual embeddings
my_reps = model.extract_representation(my_sentences, layer=MY_LAYER)

# Normalize the embeddings and calculate cosine similarities
my_reps_np = my_reps.numpy()
my_reps_norm = normalize(my_reps_np, layer=MY_LAYER, means_df=bert_means, sds_df=bert_sds)
my_sim_matrix = cosine_similarity(my_reps_norm)

# Print the result
labels = [f"Sentence {i+1}" for i in range(len(my_sentences))]
my_sim_df = pd.DataFrame(my_sim_matrix, index=labels, columns=labels).round(3)
print(my_sim_df)

<br>
<br>
<br>

## **Section 3: The geometry of argument-structure constructions**

One of the central constructions studied in psycholinguistics is the English dative alternation:
*   She gave the librarian the book (double-object, **DO**)
*   She gave the book to the librarian (prepositional-object, **PO**)

These two constructions do not mean exactly the same thing. The DO structure is prototypically associated with a "possession transfer" event, which changes who "owns" the direct object. In contrast, the PO structure is prototypically associated with a "location change" event, which changes where the direct object is. Support for this claim comes from sentences that are valid in one structure but not the other&mdash;if the two structures meant exactly the same thing, then any sentence that was acceptable in one structure would be just as acceptable in the other:


*   More acceptable in PO: I sent the kids to the countryside.
    
    (DO: \* I sent the countryside the kids)
    
    Explanation: a counteryside is inanimate and therefore cannot be in "possession"; kids are animate and therefore cannot be "owned".

*   More acceptable in DO: The gadget cost the child ten dollars.
    
    (PO: \* The gadget cost ten dollars to the child).
    
    Explanation: the DO version denotes "negation" of possession, such that the child lost money. The ten dollars did not change location, but rather changed owner, hence the lower acceptability in the PO structure.

More generally, sentences have gradient acceptability in the two structures: some are *much* more acceptable in one structure than the other, and can be said to be "prototypical" examples of that structure; others are only *slightly* more acceptable in one structure than the other; and still others sound equally acceptable in both structures. Many factors influence such acceptability, e.g., the discourse prominence and processing ease of the two nominals after the verb (their definiteness, animacy, length, etc.).

### The research question

Is BERT sensitive to the fine-grained, gradient distinctions between different DO and PO sentences? If so, we would expect these distinctions to influence the geometry of its contextual embeddings:

First, sentences that are prototypical of DO should be "far" apart in the high-dimensional space (or in the relevant subspace) from sentences that are prototypical of PO.

And second, sentences that are acceptable in both structures should be closer together, because they are not prototypical of either structure.

Here, we'll look at whether BERT's geometry tracks the gradient organization of sentences in the dative alternation. Specifically, we'll test whether this geometry mirrors human ratings of structural preference between DO and PO versions of sentences.

This section is a simplified version of the analyses in [this paper](https://aclanthology.org/2025.cxgsnlp-1.15/).

### The behavioral dataset

We'll use a subset of the **D**ative **A**lternation and **I**nformation **S**tructure (DAIS) dataset, reported in this [paper](https://aclanthology.org/2020.emnlp-main.376/).

To collect this dataset, participants were shown minimal pairs of sentences&mdash;one in a DO structure and one in a PO structure&mdash;and had to indicate their preference on a scale from 1 (PO preferred) to 100 (DO preferred). So, these preference ratings track "DO preference". The dataset contains 5,000 pairs, each with an average preference from 50 participants. We'll only use a small subset of pairs here.

### 3.1: Extracting contextual embeddings for all tokens in a sentence

To get familiar with the type of data we'll use, run the code below to extract embeddings of each token for the following sentence:

**"Bob took the meal to the man wearing the hat"**

Just for the code block below, we will extract embeddings from layer 8 (the layer does not matter, this is just for demonstration).

Inspect the resulting output.

**Note:** Make sure you run the "preliminaries" code (the first code chunk in this notebook) before running this part.


In [ ]:
LAYER = 8  # just for demonstration; the layer does not matter for our purposes here.

sentence = [("Bob took the meal to the man wearing the hat.")]

input_ids, hidden_states = model.encode_text(sentence, layer=LAYER)
tokens = model.tokenizer.convert_ids_to_tokens(input_ids[0])

print(f"Sentence: {sentence!r}")
print(f"Layer: {LAYER}\n")
print(f"{'Token':<15} First 5 values of its 768-number embedding")
for token, vector in zip(tokens, hidden_states[0]):
    print(f"{token:<15} {vector[:5].numpy().round(3)}")

**What to notice:** every sentence that BERT processes gets two extra tokens added automatically: `[CLS]` at the very start and `[SEP]` at the very end.

Both get their own full 768-number embedding at every layer, exactly like a real word does&mdash;`[CLS]` is trained to summarize the whole sentence, and `[SEP]` marks where a segment of text ends (useful when BERT is given two sentences at once, as in its original next-sentence-prediction training task; here we're only giving it one sentence, so there's just a single `[SEP]` at the end, right after the final period).

In the analyses below, we will use the `[CLS]` token as a representation of the entire sentence. There are other ways to represent an entire sentence&mdash;we will explore those in later sections.

### 3.2: Functions that extract embeddings from all layers at once

This exercise requires contextual embeddings from every one of BERT's 12 hidden layers, so we define two functions to help us in this process.

In [ ]:
# This exercise needs EVERY layer's representation at once (not just one
# layer, like Section 1 did). minicons's encode_text(sentences, layer="all")
# does exactly this in a single forward pass for the WHOLE batch of
# sentences at once (not one pass per sentence), so we reuse Section 1's
# `model` object instead of loading BERT a second time.

def cls_all_layers(sentences):
    """Runs ALL the sentences through BERT together in one batched forward
    pass via minicons's encode_text(), with layer="all" so we get back every one of BERT's layers at once;
    then, we keep [CLS] (position 0) from each layer. [CLS] is the FIRST
    token, not the last: BERT is bidirectional, so every token already
    attends to the whole sentence; [CLS] is special only because BERT's
    pretraining (next-sentence prediction) specifically trained this one
    position to summarize the sequence -- contrast this with GPT-2 later,
    which has no such token and is forced to use its LAST token instead.
    Returns shape (num_sentences, num_layers, embedding_size)."""
    input_ids, hidden_states = model.encode_text(sentences, layer="all")
               # hidden_states is a list of (n_layers) tensors, each of size (num_sentences, sentence_length, embedding_size)
    cls_per_layer = [h[:, 0, :].numpy() for h in hidden_states]
    return np.stack(cls_per_layer, axis=1)

def select_layer(all_layer_reps, layer):
    """Pull out one layer's vectors from an (num_sentences, num_layers, embedding_size) array -> (num_sentences, embedding_size).
    This is just indexing: no recomputation, since cls_all_layers() already ran the model once for the whole batch and kept every layer."""
    return all_layer_reps[:, layer, :]

### 3.3: Load the DO/PO stimulus set

This section loads the real stimulus set (and associated behavioral data) from a GitHub repository. The dataset includes many variables, but we only need three columns: `DOsentence`, `PDsentence` (the two versions of each item), and `BehavDOpreference` (the human preference score).

**Note:** What we call the PO structure is referred in the dataset as the PD structure.

In [ ]:
DOPO_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/generated_pairs_with_results.csv"

dopo_df = pd.read_csv(DOPO_URL)

# The file has several other columns (verb metadata, probability scores from several LMs)
# that we won't use in this exercise. We just look at the three we need.
relevant_cols = ["DOsentence", "PDsentence", "BehavDOpreference"]


# Display the first 5 rows
from IPython.display import display

pd.set_option("display.max_colwidth", None)  # show full sentence text (without a "..." truncation)

display(dopo_df[relevant_cols].head())
print()
print("BehavDOpreference summary:")
print(dopo_df["BehavDOpreference"].describe())


### 3.4: Bin sentences by preference, then deterministically sample a subset of them

We don't have time to process all 5000 items, so we take a smaller sample. First, we divide all the sentences into 5 preference bins of equal width, spanning 0-100: [0 20], [20 40], [40 60], [60 80], [80 100]. Then, from each bin, we sample items closest to that bin's center. So, we end up with sentences whose DO-preference ratings are closest to 10, 30, 50, 70, and 90.

This is **not random**: we're picking items near each bin's center, which keeps the 5 bins as "distinguishable" from each other as possible. It also means that every run of this notebook picks the exact same items.

In [ ]:
N_PER_BIN = 10 # number of sentences sampled per bin

def sample_by_bin(df, preference_col="BehavDOpreference", n_bins=5, n_per_bin=N_PER_BIN):
    """Deterministic, stratified sample: bin the data into n_bins equal-width bins over [0, 100]
    based preference_col, then from each bin take the n_per_bin items CLOSEST TO THAT BIN'S CENTER
    (e.g. 10, 30, 50, 70, 90 for 5 bins). This approach maximizes
    separation BETWEEN bins:  items near a bin's center are as far as
    possible from the neighboring bin's boundary, so the 5 bins' samples
    are more clearly distinguishable from each other on preference_col."""
    df = df.copy()
    bin_edges = np.linspace(0, 100, n_bins + 1)  # 5 bins means there are 5+1 = 6 "edges" defining their boundaries: 0, 20, 40, 60, 80, 100
    df["bin"] = pd.cut(df[preference_col], bins=bin_edges, labels=False, include_lowest=True)

    sampled = []
    for b in range(n_bins):
        bin_center = (bin_edges[b] + bin_edges[b + 1]) / 2
        bin_df = df[df["bin"] == b].copy()
        bin_df["dist_to_center"] = (bin_df[preference_col] - bin_center).abs()  # absolute deviation of each sentence's preference score from the bin center

        # sort by closeness to center
        bin_df = bin_df.sort_values(["dist_to_center", preference_col], kind="mergesort") # "mergesort" forces a stable sorting algorithm
        sampled.append(bin_df.head(n_per_bin).drop(columns="dist_to_center"))
    return pd.concat(sampled).reset_index(drop=True)


# Display the chosen sentences
from IPython.display import display

pd.set_option("display.max_colwidth", None)  # show full sentence text (without a "..." truncation)

sample_df = sample_by_bin(dopo_df)
display(sample_df[["DOsentence", "PDsentence", "BehavDOpreference", "bin"]])

print()
print(f"Items per bin (should be {N_PER_BIN} each):")
print(sample_df["bin"].value_counts().sort_index())

### 3.5: Extract contextual embeddings for all sentences, from all layers

This is the only extraction call for this exercise. Everything after this part just calls the two arrays created here, no matter which layer you pick.

In [ ]:
do_sentences = sample_df["DOsentence"].tolist()
po_sentences = sample_df["PDsentence"].tolist()

all_layer_do = cls_all_layers(do_sentences)
all_layer_po = cls_all_layers(po_sentences)

print(f"DO shape: {all_layer_do.shape} -- {all_layer_do.shape[0]} sentences x {all_layer_do.shape[1]} layers x {all_layer_do.shape[2]} dimensions per layer")
print(f"PO shape: {all_layer_po.shape} -- {all_layer_do.shape[0]} sentences x {all_layer_do.shape[1]} layers x {all_layer_do.shape[2]} dimensions per layer")

**Note**: There are 13 layers because the ``cls_all_layers`` function includes the static embedding layer, in addition to the 12 hidden layers.

### 3.6: Load normalization statistics, define `normalize()` (again...)

In exercise 1, we discussed the need to normalize the activation of each unit in a contextual embedding (to deal with anisotropy). The chunk below is identical to 1.7, and it is replicated here so that exercise 3 is largely "self contained".

In [ ]:
BERT_MEANS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/bert_unit_means.tsv"
BERT_SDS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/bert_unit_sds.tsv"

# Wide format, no header row: row i holds the 768 per-unit values for layer i.
bert_means = pd.read_csv(BERT_MEANS_URL, sep="\t", header=None)
bert_sds = pd.read_csv(BERT_SDS_URL, sep="\t", header=None)
print(f"Means table shape: {bert_means.shape} -- {bert_means.shape[0]} layers x {bert_means.shape[1]} units")
print(f"SDs table shape:   {bert_sds.shape} -- {bert_sds.shape[0]} layers x {bert_sds.shape[1]} units")

# A function that takes contextual embeddings, and normalizes the value of each hidden unit relative
# tot he means and SDs in the variables above (computed based on contextual embeddings for a section of COCA)
def normalize(vectors, layer, means_df, sds_df):
    """Z-score each of the 768 dimensions using external corpus statistics
    for this layer. means_df/sds_df are WIDE-format tables loaded with header=None:
    row i holds per-unit values for layer i (row 0 = static embeddings;
    rows 1-12 = after each transformer block). Pull out the row
    for the requested layer, then apply it column-wise."""
    layer_means = means_df.iloc[layer].values
    layer_sds = sds_df.iloc[layer].values
    return (vectors - layer_means) / layer_sds

### 3.7: Define similarity function

Let's define a function that will compute the cosine similarity for each pair of DO-PO sentences.

In [ ]:
from sklearn.metrics.pairwise import paired_cosine_distances

def paired_cosine_similarity(A, B):
    """Row-wise cosine similarity between two same-shape (num_sentences, embedding_size)
    arrays: A[i] compared to B[i]. sklearn's paired_cosine_distances computes
    this row-wise comparison directly (unlike the cosine_similarity() functions, which returns
    a full every-pair-against-every-pair matrix); cosine similarity = 1 - cosine distance."""
    return 1 - paired_cosine_distances(A, B)

### 3.8: The main analysis: evaluate and plot DO-PO similarities

Now we can ask: if a DO sentence is prototypical of its construction, does BERT represent it as more distinct from its PO version? If both sentences in a DO-PO pair are acceptable (neither is "prototypical"), does BERT represent them as more similar?

First, choose a layer you'd like to analyze.
The code will then normalize the embeddings of all DO/PO pairs at that chosen layer, and compute a cosine similarity score per pair.
Finally, the code will plot&mdash;for each of the 5 bins&mdash;the average  similarity across sentence pairs from that bin.

In [ ]:
# EDIT ME: Choose a hidden layer between 1 and 12 (0 is the static embedding layer)
LAYER = 9;

# Extract contextual embeddings from the relevant layer
do_layer = select_layer(all_layer_do, LAYER)
po_layer = select_layer(all_layer_po, LAYER)

# Normalize the embeddings
do_layer_norm = normalize(do_layer, LAYER, bert_means, bert_sds)
po_layer_norm = normalize(po_layer, LAYER, bert_means, bert_sds)

# Compute similarities between every DO-PO pair
sample_df["do_po_similarity"] = paired_cosine_similarity(do_layer_norm, po_layer_norm)

# Compuate average similarity per bin
bin_avg_similarity = sample_df.groupby("bin")["do_po_similarity"].mean()
bin_se_similarity = sample_df.groupby("bin")["do_po_similarity"].sem()  # standard error = std / sqrt(n)
bin_labels = ["0-20", "20-40", "40-60", "60-80", "80-100"]


# Print the list of sentence pairs + their similarities
from IPython.display import display
display(sample_df.sort_values("BehavDOpreference")[
    ["DOsentence", "PDsentence", "BehavDOpreference", "bin", "do_po_similarity"]
])

# Plot the results
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.bar(bin_labels, bin_avg_similarity.values, yerr=bin_se_similarity.values, capsize=5, color="#1f77b4")
plt.title("Average DO-PO similarity by human preference bin")
plt.xlabel("Human DO-preference bin")
plt.ylabel("Average DO-PO cosine similarity")

# Zoom the y-axis in on the actual range of values (plus a small margin), instead of starting at 0.
# With cosine similarities this close together, starting at 0 might visually flatten out real differences between bins.
lowest_point = (bin_avg_similarity - bin_se_similarity).min()
highest_point = (bin_avg_similarity + bin_se_similarity).max()
margin = 0.1 * (highest_point - lowest_point)
plt.ylim(lowest_point - margin, highest_point + margin)

plt.show()

In [ ]:
#@title 3.9: Do high-preference pairs (strong preference for either DO or PO) show a clear pattern of being more or less similar than no-preference (middle-of-the-road) pairs?
your_answer = "High-preference pairs are LESS similar" #@param ["High-preference pairs are LESS similar", "High-preference pairs are MORE similar", "No clear trend"]

# Live-graded
diff_2_0 = bin_avg_similarity.loc[2] - bin_avg_similarity.loc[0]
diff_2_4 = bin_avg_similarity.loc[2] - bin_avg_similarity.loc[4]

se_diff_2_0 = np.sqrt(bin_se_similarity.loc[2]**2 + bin_se_similarity.loc[0]**2)
se_diff_2_4 = np.sqrt(bin_se_similarity.loc[2]**2 + bin_se_similarity.loc[4]**2)

z_2_0 = diff_2_0 / se_diff_2_0
z_2_4 = diff_2_4 / se_diff_2_4

SEPARATION_THRESHOLD = 1.0  # bins must be at least 1 combined SE apart to count as "separated" (lenient)

if z_2_0 > SEPARATION_THRESHOLD and z_2_4 > SEPARATION_THRESHOLD:
    actual_ans = "High-preference pairs are LESS similar"
elif z_2_0 < -SEPARATION_THRESHOLD and z_2_4 < -SEPARATION_THRESHOLD:
    actual_ans = "High-preference pairs are MORE similar"
else:
    actual_ans = "No clear trend"

if your_answer == actual_ans:
    print(f"✅ Correct! bin2 vs bin0: {diff_2_0:.3f} ({z_2_0:.2f} combined SEs apart); bin2 vs bin4: {diff_2_4:.3f} ({z_2_4:.2f} combined SEs apart) -> {actual_ans}")
else:
    print(f"❌ Not quite. bin2 vs bin0: {diff_2_0:.3f} ({z_2_0:.2f} combined SEs apart); bin2 vs bin4: {diff_2_4:.3f} ({z_2_4:.2f} combined SEs apart) -> {actual_ans}")

**What to notice:** examine the pattern in different layers (by changing ``LAYER`` in 3.9). Which layers seem to represent sentences in ways that are more aligned with linguistic theory?

What happens if we run such an experiment, and find that the effect of interest is only observed in one or two layers? How do we interepret that finding? Is it a fluke, or something meaningful? How would you know?
<br>
<br>
<br>

## **Section 4: Probing for information inside contextual embeddings**

So far we've been measuring similarities between embeddings. Indirectly, these similarities reveal something about **the information that is encoded in these embeddings**. But we can ask this question even more directly, using an approach called **probing**.

Probing asks: "does information about linguistic feature X exist inside the embeddings?". It does so by training a "probing classifer"&mdash;a small statistical model&mdash;to predict the value of X from the embedding (or "extract" the value of X from the embedding). For example, we can ask:


*   If we take the embedding of a verb, can a classifier predict whether that verb is singular or plural?
*   If we take the embedding of a verb's direct object, can a classifier predict how plausible that direct object is (e.g., for a verb like "eat", "cake" is more plausible than "sand", which is more plausible than "democracy").
*   If we take the embedding of a noun, can a classifier predict whether it's the direct object or indirect object of a verb?

And so on.

If the classifier succeeds in predicting linguistic feature X, then we can say that the LM's representation implicitly encodes (some approximation of) this feature, even though the LM was never explicitly trained to do so!

### The current exercise ###
In this exercise, we will ask: **do contextual embeddings implicitly represent whether a sentence is grammatical?**

Using a probing approach, this question becomes: can a simple classifier predict, based on contextual embeddings, whether a sentence is grammatical vs. ungrammatical? For this purpose, we will assume that grammaticality is binary (a sentence is either grammatical or not), even though many contemporary theories may treat grammaticality as a more gradient property of a sentence.

For this exercise, we will also switch LMs: from *BERT*, with its bidirectional attention, to *GPT2-small*, which is a unidirectional decoder LM. In GPT2-small, each token can only attend to past tokens and to itself, but not to future tokens. We use GPT2-small so that you get experience working with such LMs, which are more common in psycholniguistic research (because they provide a closer analog to incremental language processing in humans).

This section is a simplified version of the analyses in [this paper](https://aclanthology.org/2026.acl-long.686/).

### 4.1: Load GPT2-small and inspect its architecture

In [ ]:
from minicons import cwe

# Same minicons.cwe.CWE wrapper that Section 1 used for BERT, now pointed at GPT-2.
# You can delete device=device below; in that case, minicons will just use cpu
model_gpt2 = cwe.CWE("gpt2", device=device)
print("GPT-2 loaded and ready.\n")

gpt2_config = model_gpt2.model.config

n_layers_gpt2 = gpt2_config.num_hidden_layers           # number of transformer blocks stacked on top of each other
hidden_size_gpt2 = gpt2_config.hidden_size              # length of the embedding representing each token
n_heads_gpt2 = gpt2_config.num_attention_heads          # how many "attention heads" in each block
vocab_size_gpt2 = gpt2_config.vocab_size                # how many distinct tokens are in GPT2's vocabulary
n_params_gpt2 = sum(p.numel() for p in model_gpt2.model.parameters())  # total count of the model's learned numbers

print(f"Transformer blocks (\"layers\"):          {n_layers_gpt2}")
print(f"Embedding size per token (hidden size): {hidden_size_gpt2}")
print(f"Attention heads per block:              {n_heads_gpt2}")
print(f"Vocabulary size:                        {vocab_size_gpt2:,} tokens")
print(f"Total parameters:                       {n_params_gpt2:,} (approximately {n_params_gpt2 / 1e6:.0f} million)")

**What to note:** GPT2's usual vocabulary is 50,257 tokens; here, we see one more token. This is because ``minicons`` adds a "padding" token (see comment below), which the original GPT2 doesn't have.


### 4.2: Load a mini-corpus of real vs. corrupted sentences
To probe whether the contextual embeddings of GPT2 implicitly represent grammaticality, we will train a simple classifier that takes a sentence embedding as input, and predicts whether that sentence is grammatical or not.

We train this classifer on a mini-corpus which consists of real sentences and, for each of them, an *artificially corrupted* version. To create this corpus, I sampled 3000 sentences from the Brown corpus and the Project Gutenberg corpus (from Python's `NLTK` platform). After tokenizing the sentences, I created a corrupted version of each sentence. Corruptions occurred in one of 3 ways: inserting 1-5 random tokens, deleting 1-5 tokens, or shuffling the order of 5 contiguous tokens.

Note that these corruptions do not target specific grammatical regularities. They are intended to create generic, "something is off here" violations. Moreover, not all corruptions will necessarily result in an ungrammatical sentence (e.g., if we take *"the big dog jumped"* and delete *"big"*, the resulting sentence remains well-formed). Still, we will assume that most of our corrupted sentences are ungrammatical, and that's good enough.

The code below loads these sentences. Each row is one sentence, and has a label. `label=1` means it's a real (and, we assume, grammatical) sentence; `label=0` means it's been corrupted.



In [ ]:
CORRUPT_CORPUS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/WangLikeGrammaticalityStimuli.csv"

corpus_df = pd.read_csv(CORRUPT_CORPUS_URL)

# Display some example sentences: 2 examples of each corruption type
from IPython.display import display
pd.set_option("display.max_colwidth", None)  # show full sentence text (without a "..." truncation)

example_pair_ids = corpus_df.groupby("perturbation_type").head(2)["pair_id"]
examples = corpus_df[corpus_df["pair_id"].isin(example_pair_ids)].sort_values(["pair_id", "label"], ascending=[True, False])
print("Example sentences:")
display(examples[["pair_id", "sentence", "label", "perturbation_type"]])

print()
print("Sentences per perturbation type (NaN = real, uncorrupted sentence):")
print(corpus_df["perturbation_type"].value_counts(dropna=False))

### 4.3: GPT2's tokenization conventions

Before extracting anything, let's look at how GPT2 tokenizes a couple of sentences from this corpus (the same way we checked BERT's tokenization back in section 1.5).

In [ ]:
for sentence in corpus_df["sentence"].iloc[[2002, 4002]]:  # I chose two specific sentences for illustrative purposes
    print(sentence)
    print(f"  -> tokens: {model_gpt2.tokenizer.tokenize(sentence)}")
    print()


**What to notice:** A `Ġ` prefix marks a token that *starts* a new word. It is GPT2's way of encoding "this token was preceded by a space." This marking is the opposite convention from BERT's `##`, which marked *continuations* instead of *beginnings*.

A token *without* a `Ġ` (other than the very first token in a sentence, which never gets one, because nothing precedes it) is a continuation of the previous token, not a new word. Several words above split into multiple tokens this way&mdash;e.g. "lyricist" becomes `lyric` + `ist`, and "uncompromising" becomes `uncomp` + `romising`.

Just like BERT, GPT2 has separate embeddings for the word-initial vs. continuation versions of the same string. Of the ~33,100 word-initial tokens in GPT2's vocabulary, about 26% have a matching continuation-form token.

### 4.4: Helper function to subsample the corpus
The full corpus has 6,000 sentences&mdash;more than we need for this workshop, and more than is comfortable to extract live on CPU. We'll take a subset of 1,200 sentences, which will be:

*   **balanced**: equal number of grammatical vs. corrupted sentences (obtained by sampling minimal pairs)

*   **stratified**: a (nearly) even mix of the three corruption types. Stratifying by corruption type matters&mdash;if the sentences all shared one corruption type, the classifier trained to distinguish embeddings of real vs. corrupted sentences might simply learn to detect that one specific type, rather than learn to detect ungrammaticality more generally.

The helper function below performs this sub-sampling.


In [ ]:
def stratified_subsample(df, n_per_class, group_col="perturbation_type"):
    """Deterministically build a small, balanced subsample:
    pick n_per_class corrupted sentences, spread as evenly as
    possible across each corruption type in group_col, then pull each
    chosen sentence's matching grammatical sentence (same pair_id)
    alongside it.
    Note: n_per_class refers to the "grammatical" vs. "ungrammatical"
    classes, not to corruption types."""
    corrupted = df[df["label"] == 0].copy()

    # rank each corrupted sentence within its own corruption type (0, 1, 2, ...)
    corrupted["rank_in_type"] = corrupted.groupby(group_col).cumcount()

    # Split n_per_class across the corruption types as evenly as possible:
    # if n_per_class doesn't divide evenly, give the leftover remainder to the first
    # few types (in sorted order) instead of just dropping it, so the total
    # always adds up to exactly n_per_class.
    types = sorted(corrupted[group_col].unique())
    n_types = len(types)
    base, remainder = divmod(n_per_class, n_types)
    n_per_type = {t: base + (1 if i < remainder else 0) for i, t in enumerate(types)}

    corrupted["type_quota"] = corrupted[group_col].map(n_per_type)
    chosen_pair_ids = corrupted.loc[corrupted["rank_in_type"] < corrupted["type_quota"], "pair_id"]
    return df[df["pair_id"].isin(chosen_pair_ids)].reset_index(drop=True)

### 4.5: Helper function for extracting "sentence embeddings"

If a LM represents each token as a contextual embedding, how do we get an embedding for a *full sentence*? In BERT, we used the ``[CLS]`` token, but GPT2-small doesn't have such a token.

There are two common ways to get a full-sentence embedding:
1. Use the embedding of the last token&mdash;often, a punctuation like ``.`` or ``?``. This token is the only one that has attended to all other tokens in the sentence, and thus can get information from all previous words.
2. Average the embeddings of all tokens together (an approach called mean-pooling).

There is currently no consensus on which approach is better&mdash;different studies use one or the other. Here, you will be able to choose which approach to work with.

Below, we define helper functions to assist in creating sentence embeddings. In Section 3, all sentences were fed to the LM in one, single batch. This is OK for the few dozen sentences we used, but here we'll eventually use over 1000 sentences. Stuffing all of them into one single batch at once would use too much memory. So, the helpers below process sentences in chunks (a chunk = a smaller group of sentences run through the model together&mdash;here, 32 at a time).
<br>
<br>

###Important note: padding
When you batch several sentences together and pass them to the LM, shorter ones get "padded" at the end with filler tokens, so every sentence in the batch has the same length. Padding creates two separate traps:

**1. Contaminated representations.**

If you forget to pass an `attention_mask` into the LM, every token&mdash;including whichever one you're trying to extract&mdash;will attend to the padding filler tokens too, because the model has no way of knowing they aren't real words. This is a problem with bidirectional LMs, which can attend to the future; unidirectional LMs cannot do that, so "real" tokens from the sentence cannot attend to the "padding" tokens that come later. Attending to padding tokens silently changes the resulting embeddings, and the amount of contamination depends on how much padding a given sentence happened to get, which depends on which other sentences it was batched with.

Luckily, `minicons`'s `encode_text()` solves this problem for you automatically: it passes an `attention_mask` into the LM, so the LM's own attention computation (at evert layer) ignores padding.

**2. Grabbing the wrong token.**

`encode_text()` still returns an embedding for *every* token, including padding ones. Instead of removing them, it zeroes them out (so the embedding is just a list of 0s). If you grabbed, say, the embedding for "the last token in the array", you would often get an all-zero padding embedding instead of the sentence's real last word. Alternatively, if you averaged the embeddings of all tokens in the sequence, you would include those padded tokens (a bunch of zero-embeddings, which would pull the average activations towards 0).
<br>
<br>
For this reason, `last_token_reps()` and `mean_pool_reps()` in this notebook manually reconstruct the attention mask and compute each sentence's true length themselves.

**When you work with LMs on your own, the best solution may be to write a loop that feeds each sentence separately to the LM, without batching.**

In [ ]:
from tqdm import tqdm
    # tqdm is for showing a progress bar. Extracting thousands
    # of sentences on CPU can take a couple of minutes, and the bar
    # will tell you it's working rather than looking frozen.


def last_token_reps(sentences, layer, batch_size=32):
    """Last-token (or last-word) extraction via minicons. Sentences are
    fed in batches. minicons always pads on the RIGHT, so every sentence's
    real tokens start at position 0. The attention_mask (reconstructed from
    which positions aren't the pad token) tells us where each sentence's
    real content ends, so we can find the TRUE last token.
    The function returns the embedding of the single last token, which is the
    token that has attended to every earlier word in the sentence."""
    reps = []

    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i + batch_size]
        input_ids, hidden_states = model_gpt2.encode_text(batch, layer=layer)
        attention_mask = (input_ids != model_gpt2.tokenizer.pad_token_id).numpy()
        hidden_states_np = hidden_states.numpy()
        last_idx = attention_mask.sum(axis=1) - 1  # true last token per row, ignoring padding

        batch_reps = []
        for row in range(hidden_states_np.shape[0]):  # loop through the embeddings of each sentence
            end = last_idx[row]
            batch_reps.append(hidden_states_np[row, end])
        reps.append(np.stack(batch_reps))
    return np.concatenate(reps, axis=0)


def mean_pool_reps(sentences, layer, batch_size=32):
    """Same idea as last_token_reps, but averaging every REAL (non-padding)
    token's embedding instead of taking just the last token.
    Padded positions are zeroed-out using the reconstructed
    attention_mask, then the sum of embeddings is divided by its TRUE
    token count (not the padded length)."""
    reps = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i + batch_size]
        input_ids, hidden_states = model_gpt2.encode_text(batch, layer=layer)
        hidden_states_np = hidden_states.numpy()
        attention_mask = (input_ids != model_gpt2.tokenizer.pad_token_id).numpy()[:, :, None]
            # "None" = we add a 3rd dimension of length 1, so that the multiplication below works
            # (this is related to how NumPy operates)
        summed = (hidden_states_np * attention_mask).sum(axis=1)
        n_real_tokens = attention_mask.sum(axis=1)
        reps.append(summed / n_real_tokens)
    return np.concatenate(reps, axis=0)


def extract_reps(sentences, layer, pooling_method, batch_size=32):
    """ This is the one function the rest of the notebook calls. It routes to one
    of the two functions above (whichever pooling function is selected by
    pooling_method)."""
    if pooling_method == "last_token":
        return last_token_reps(sentences, layer, batch_size=batch_size)
    elif pooling_method == "mean_pool":
        return mean_pool_reps(sentences, layer, batch_size=batch_size)
    else:
        raise ValueError(f"Unknown pooling_method: {pooling_method!r} (expected 'last_token' or 'mean_pool')")

print("extract_reps() ready: currently supports 'last_token' and 'mean_pool'.")

### 4.6: Extracting sentence embeddings

For each sentence in the corpus, we'll extract a contextual embedding from a **single layer** in GPT2. You can choose which layer you'd like to use; as a default, we'll use layer 8.

For now, we'll represent a sentence as the contextual embedding of the last token (the punctuation). Later, you can experiment with using mean-pooling instead.

The classifer that we train later will learn how to take as input a sentence embedding and, based on that embedding, produce as output the probability that the sentence is grammatical (vs. ungrammatical / corrupted). To train this classifer, we need to give it a set of examples: sentence embeddings, each paired with a label of "grammatical" or "ungrammatical". The code below extracts these examples for our whole 1,200-sentence sample at once. The embeddings are called ``X`` and the labels are called ``y``.

In [ ]:
# EDIT ME: CHOOSE A LAYER AND A POOLING METHOD
GPT2_LAYER = 8  # 1-12 = hidden layers, 0 = static embedding
POOLING_METHOD = "last_token"  # options: "last_token" vs. "mean_pool"

# Build one balanced, stratified sample from the WHOLE corpus: 600 minimal
# pairs (1200 sentences total).
sample_df = stratified_subsample(corpus_df, n_per_class=600)

X_raw = extract_reps(sample_df["sentence"].tolist(), layer=GPT2_LAYER, pooling_method=POOLING_METHOD)
y = sample_df["label"].values

print(f"\nShape: {X_raw.shape} -- {X_raw.shape[0]} sentences ({(y == 1).sum()} grammatical, {(y == 0).sum()} corrupted), each a {X_raw.shape[1]}-number embedding")

### 4.7: Load normalization statistics, define ``normalize()`` and apply it

Same idea as Section 3: we will *z*-score the embeddings using the mean + SD of each hidden unit's activations, evaluated based on a an external corpus (this time, the statistics are compute dfor GPT2, not BERT).

In [ ]:
GPT2_MEANS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/gpt2_unit_means.tsv"
GPT2_SDS_URL = "https://raw.githubusercontent.com/idanblank/LM-embedding-workshop-data/refs/heads/main/gpt2_unit_sds.tsv"

# Wide format, no header row: row i holds the 768 per-unit values for layer i.
gpt2_means = pd.read_csv(GPT2_MEANS_URL, sep="\t", header=None)
gpt2_sds = pd.read_csv(GPT2_SDS_URL, sep="\t", header=None)
print(f"Means table shape: {gpt2_means.shape} -- {gpt2_means.shape[0]} layers x {gpt2_means.shape[1]} units")
print(f"SDs table shape:   {gpt2_sds.shape} -- {gpt2_sds.shape[0]} layers x {gpt2_sds.shape[1]} units")

# A function that takes contextual embeddings, and normalizes the value of each hidden unit relative
# to the means and SDs in the variables above (computed based on contextual embeddings for a section of COCA)
def normalize(vectors, layer, means_df, sds_df):
    """Z-score each of N dimensions using external corpus statistics
    for this layer. means_df/sds_df are WIDE-format tables loaded with header=None:
    row i holds per-unit values for layer i (row 0 = static embeddings;
    rows 1-12 = after each transformer block). Pull out the row
    for the requested layer, then apply it column-wise."""
    layer_means = means_df.iloc[layer].values
    layer_sds = sds_df.iloc[layer].values
    return (vectors - layer_means) / layer_sds

# Apply the function
X_norm = normalize(X_raw, GPT2_LAYER, gpt2_means, gpt2_sds)

### 4.8: The probe&mdash;a logistic-regression classifier

Our probe&mdash;the classifier that will attempt to predict whether a sentence is grammatical or not based on its embedding&mdash;will be a **logistic regression**. This is a simple statistical model, which learns a weighted combination of the 768 numbers in the embedding, in order to predict grammaticality. In practice, even though logistic regression is trained on a dataset with binary labels (grammatical vs. ungrammatical), it actually outputs a continuous prediction: the *probability* that a sentence is grammatical.

When training a logistic regression model with so many variables (here: 768), we want to discourage it from fitting the training data *too* well (over-fitting). Such a model might not generalize well to new sentences that it has not seen during training. One way to apply regularization is via a penalty on the size of the classifier weights&mdash;driving the logistic regression model to avoid weights that are "too large" (because large weights often let a model "bend" itself around the idiosyncrasies of the specific training examples it sees, rather than learn a general pattern).

In the code below, this regularization is (inversely) controlled by a parameter `C`: a smaller value of `C` means a stronger penalty. How do we choose the value of `C`? We try a small set of `C` values: for each value, we train a logistic regression using a subset of our corpus, and then score how well the model generalizes to the remaining sentences. In other words, we ask: how well can our probe classify the grammaticality of new sentences that it hasn't seen? We then choose the value of ``C`` for which our probe shows the best performance.

For this purpose, we need to split our examples of (embedding, label) into two subsets:

*   **Training** subset: used to train the classifier (here, fit the logistic regression weights).

*   **Development** subset: used only to check how well a given classifier (trained with a specific value of `C`) generalizes to unseen sentences.

This split happens automatically, inside `split_train_dev()` below. This split is deterministic (same split every time you run it) and stratified (a nearly even mix of all 3 corruption types for each subset). It keeps each grammatical/corrupted pair together in the same subset, so we never train on one half of a pair and evaluate on the other half.




In [ ]:
from sklearn.linear_model import LogisticRegression  # sklearn is a machine learning library, which includes logistic regression


def split_train_dev(y, pair_ids, perturbation_types, train_frac=0.8):
    """Deterministically splits examples into train/dev, stratified by
    perturbation_type, and split by minimal pair (so a grammatical sentence
    and its matching corrupted counterpart always land in the same
    subset), with both train and dev ending up with a proportional mix
    of every corruption type. Returns a boolean array: True for rows
    assigned to train, False for dev."""
    # perturbation_type is only non-NaN for corrupted (label=0) sentences, so we
    # use those to generate the split, and then apply it to BOTH sentences in each minimal pair.
    corrupted_mask = (y == 0)
    corrupted_pairs = pd.DataFrame({
        "pair_id": pair_ids[corrupted_mask],
        "perturbation_type": perturbation_types[corrupted_mask],
    })

    # Choose which minimal pairs (defined by their IDs) go to train vs. dev
    train_pair_ids = []
    for ptype, group in corrupted_pairs.groupby("perturbation_type"):
        pids = sorted(group["pair_id"])
        n_train = int(len(pids) * train_frac)
        train_pair_ids.extend(pids[:n_train])
    train_pair_ids = set(train_pair_ids)

    return np.array([pid in train_pair_ids for pid in pair_ids])


def fit_and_tune_probe(X, y, pair_ids, perturbation_types, C_values, train_frac=0.8):
    """Uses split_train_dev() to deterministically split (X=embeddings,
    y=labels) examples into train/dev. The default split is 80%(train)/20%(dev).
    The code loops over C_values, fitting a logistic regression probe on the train subset
    and scoring it on the dev subset. It returns whichever C generalized best."""
    is_train = split_train_dev(y, pair_ids, perturbation_types, train_frac=train_frac)
    X_train, y_train = X[is_train], y[is_train]
    X_dev, y_dev = X[~is_train], y[~is_train]

    # Iterate over values of C, score each logistic regression model by accuracy on dev subset
    tuning_results = []
    for C in C_values:
        clf_candidate = LogisticRegression(C=C, max_iter=2000)
        clf_candidate.fit(X_train, y_train)
        dev_acc = clf_candidate.score(X_dev, y_dev)
        tuning_results.append((C, dev_acc))

    # Choose the best C, run logistic regression with its value
    # Once we find C, we fit the logistic regression to ALL data (train + dev)
    tuning_df = pd.DataFrame(tuning_results, columns=["C", "dev_accuracy"])
    best_C, best_dev_acc = max(tuning_results, key=lambda r: r[1])
    clf = LogisticRegression(C=best_C, max_iter=2000).fit(X, y)

    return clf, best_C, best_dev_acc, tuning_df


###4.9 Train the probe
We evaluate the probe's performance on the development subset of our corpus. To this end, we feed the embedding of each sentence to the logistic regression classifier, and get the probability that the sentence is grammatical.

If the probability is > 50%, we treat it as a "grammatical" classification.

If the probability is <= 50%,  we treat it as an "ungrammatical" classification.

Then, we find the % of sentences for which the probe gave the correct classification.

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
clf, best_C, best_dev_acc, tuning_df = fit_and_tune_probe(
    X_norm, y, sample_df["pair_id"].values, sample_df["perturbation_type"].values, C_values, train_frac=0.8
)

print(tuning_df)
print(f"\nBest C: {best_C}, dev accuracy: {best_dev_acc:.3f}")

### 4.10: Sanity-check your dev accuracy

Type in the best dev accuracy YOU saw printed in Block 4.9 (as a percentage, e.g. `82.5`, without the % mark).

In [ ]:
dev_accuracy_percent = 80.4 #@param {type:"number"}

if dev_accuracy_percent < 1:
    print(f"Make sure you type the accuracy as a percentage: 60, not 0.6")
elif abs(dev_accuracy_percent - 50) <= 5:
    print(f"Your reported dev accuracy ({dev_accuracy_percent}%) is close to 50% -- chance level for a two-way choice.")
    print("This is worth flagging to the instructor; something may have gone wrong upstream.")
else:
    print(f"Your reported dev accuracy ({dev_accuracy_percent}%) is comfortably above chance -- looks reasonable, moving on.")

### 4.11: Stress-test the probe

We trained our probe on synthetic pairs of grammatical vs. corrupted sentences. What did the probe learn? Could it work on more tightly-controlled minimal pairs, with sentences whose ungrammaticality results from violating specific grammatical regularities? Here, we will test the probe on the types of minimal pairs that are familiar from studies of next-word predictions in LMs&mdash;benchmarks for testing grammatical knowledge.

The sentences we will use are completely different from what the probe has been trained on, and come from the **BLiMP** benchmark (see [here](https://github.com/alexwarstadt/blimp)). This benchmark consists of human-curated minimal pairs of grammatical vs. ungrammatical sentences; different pairs target different linguistic phenomena, such as subject-verb agreement, negative polarity licensing, etc.

If the classifier still does well on these sentences, that's evidence that GPT2's embeddings encode something general about grammaticality, not just the specific types of corruption we used for our original corpus.
<br>
<br>

###Load BLiMP
The code below loads minimal pairs targeting 6 linguistic phenomena from BLiMP, and pools them into one mixed set. To keep our evaluation quick, we only include 300 sentences total (25 pairs per phenomenon).


In [ ]:
!pip install datasets -q
from datasets import load_dataset

# The 6 phenomena below were chosen at random
blimp_phenomena = [
    "determiner_noun_agreement_1",
    "principle_A_domain_3",
    "irregular_plural_subject_verb_agreement_1",
    "wh_vs_that_with_gap",
    "npi_present_2",
    "regular_plural_subject_verb_agreement_1",
]

N_PAIRS_PER_PHENOMENON = 25  # 6 phenomena x 25 pairs x 2 sentences/pair = 300 sentences total

# Load minimal pairs for each phenomenon, and pool them all together
blimp_parts = []
for phenomenon in blimp_phenomena:
    ds = load_dataset("nyu-mll/blimp", phenomenon, split="train").select(range(N_PAIRS_PER_PHENOMENON)).to_pandas()
    ds["phenomenon"] = phenomenon
    blimp_parts.append(ds)

blimp_df = pd.concat(blimp_parts, ignore_index=True)

# Print some examples
from IPython.display import display
pd.set_option("display.max_colwidth", None)  # show full sentence text (without a "..." truncation)
examples = blimp_df.groupby("phenomenon").head(2)
display(examples[["phenomenon", "sentence_good", "sentence_bad"]])

print()
print("Sentences per phenomenon:")
print(blimp_df["phenomenon"].value_counts())

### 4.12: Extract and normalize embeddings of BLiMP minimal pairs

Same helper functions as before, this time applied to BLiMP's `sentence_good`/`sentence_bad` columns, at the same layer we trained the probe on.

In [ ]:
X_blimp_good_raw = extract_reps(blimp_df["sentence_good"].tolist(), layer=GPT2_LAYER, pooling_method=POOLING_METHOD)
X_blimp_bad_raw = extract_reps(blimp_df["sentence_bad"].tolist(), layer=GPT2_LAYER, pooling_method=POOLING_METHOD)

X_blimp_good_norm = normalize(X_blimp_good_raw, GPT2_LAYER, gpt2_means, gpt2_sds)
X_blimp_bad_norm = normalize(X_blimp_bad_raw, GPT2_LAYER, gpt2_means, gpt2_sds)

print()
print(f"BLiMP grammatical shape:   {X_blimp_good_norm.shape}")
print(f"BLiMP ungrammatical shape: {X_blimp_bad_norm.shape}")

### 4.13: Evaluate the probe on BLiMP sentences

Here, our scoring of the probe will be different than the scoring we used on our original corpus. Remember that our probe takes a sentence embedding, and returns the probability that the sentence is grammatical. So, for each minimal pair, we treat the probe as correct if it assigns a *higher* probability to the grammatical sentence than to the ungrammatical sentence.

In [ ]:
# Extract the probe's estimated probability that each sentence belongs to the "grammatical" class
scores_blimp_good = clf.predict_proba(X_blimp_good_norm)[:, 1]
scores_blimp_bad = clf.predict_proba(X_blimp_bad_norm)[:, 1]

print("A few example scores:")
for i in range(5):
    print(f"  grammatical:   {scores_blimp_good[i]:.3f}  \"{blimp_df['sentence_good'][i]}\"")
    print(f"  ungrammatical: {scores_blimp_bad[i]:.3f}  \"{blimp_df['sentence_bad'][i]}\"\n")

print(f"Mean probability for grammatical sentences:   {scores_blimp_good.mean():.3f}")
print(f"Mean probability for ungrammatical sentences: {scores_blimp_bad.mean():.3f}\n")

# Helper function for scoring accuracy
def pairwise_accuracy(grammatical_scores, ungrammatical_scores):
    """grammatical_scores[i] and ungrammatical_scores[i] must be a matched pair."""
    return float(np.mean(np.array(grammatical_scores) > np.array(ungrammatical_scores)))

# Apply the scoring function
acc_logistic_blimp = pairwise_accuracy(scores_blimp_good, scores_blimp_bad)
print(f"PAIRWISE ACCURACY: {acc_logistic_blimp:.1%}")


**What to notice:** this accuracy reflects a probe that was never trained on BLiMP, or even on real "grammatical-violation" data at all&mdash;it was only trained on our synthetic corpus of real sentences vs. corrupted ones (insertions/deletions/shuffles). A pairwise accuracy well above 50% is evidence that GPT2's embeddings implicitly encode, to some extent, sentence grammaticality.

### 4.14: Histogram of probabilities
Plot a histogram of the probabilities that the probe gives to grammatical vs. ungrammatical sentences from BLiMP.

In [ ]:
import matplotlib.pyplot as plt

def plot_score_histograms(good_scores, bad_scores, title, xlabel):
    """Overlaid histograms comparing a grammatical-side score distribution
    to an ungrammatical-side one. Shared here so Block 6.2 can reuse it
    for the mass-mean probe's scores, without redefining plotting logic."""
    plt.figure(figsize=(7, 5))
    plt.hist(good_scores, bins=20, alpha=0.6, label="Grammatical", color="#619CFF")
    plt.hist(bad_scores, bins=20, alpha=0.5, label="Ungrammatical", color="#F8766D")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Number of sentences")
    plt.legend()
    plt.show()


plot_score_histograms(
    scores_blimp_good,
    scores_blimp_bad,
    title="Probe evaluated on BLiMP",
    xlabel="P(grammatical), logistic regression classifier",
)

In [ ]:
#@title 4.15: Question: How separated are the grammatical vs. ungrammatical distributions in your histogram?
your_answer = "Some separation, but notable overlap" #@param ["Clearly separated (little overlap)", "Some separation, but notable overlap", "Mostly overlapping"]

# Live grading: this is a visual-judgment question, so we print the
# actual computed effect size alongside the user's answer.
pooled_std_415 = np.sqrt((scores_blimp_good.std() ** 2 + scores_blimp_bad.std() ** 2) / 2)
cohens_d_415 = (scores_blimp_good.mean() - scores_blimp_bad.mean()) / pooled_std_415

if cohens_d_415 > 0.8:
    actual_415 = "Clearly separated (little overlap)"
elif cohens_d_415 >= 0.3:
    actual_415 = "Some separation, but notable overlap"
else:
    actual_415 = "Mostly overlapping"

print(f"Computed Cohen's d: {cohens_d_415:.2f}")
print(f"Your answer: \"{your_answer}\" | Based on effect size: \"{actual_415}\"")
if your_answer == actual_415:
    print("✅ Your answer matches the computed effect size.")
else:
    print("ℹ️  Your answer is different than the computed effect size")
    print("This is probably a difference between visual inspection and quantitative statistics...")

<br>
<br>
<br>

##**Section 5: Another probing method (very optional)**

### 5.1: Mass-mean probing

Below, you can try a different type of probe (not logistic regression), which is even simpler: a **mass-mean probe**. To create this probe, we do the following:

1.   Average the embeddings of all the grammatical sentences together
2.   Average the embeddings of all the ungrammatical (corrupted) sentences together
3.   Use the line connecting those two averages as a "grammaticality scale" (or "grammaticality direction" in the high-dimensional embedding space).

The code below creates this probe. In this case, there is no regularization, so we train on all 1200 sentences (600 minimal pairs).

In [ ]:
mean_grammatical = X_norm[y == 1].mean(axis=0)
mean_corrupted = X_norm[y == 0].mean(axis=0)
mass_mean_direction = mean_grammatical - mean_corrupted
mass_mean_midpoint = (mean_grammatical + mean_corrupted) / 2   # the midpoint of the grammaticality direction, for plotting

print(f"Computed mass-mean direction from {len(y)} training sentences ({(y == 1).sum()} grammatical, {(y == 0).sum()} corrupted).")
print(f"Shape of the mass-mean direction: {mass_mean_direction.shape} (should be the same as the embedding size of GPT2)")


### 5.2: Evaluate the probe on BLiMP, calculate pairwise accuracy, and plot a histogram

We reuse `pairwise_accuracy()` and `plot_score_histograms()`, unchanged, from Section 4.

In [ ]:
# A helper function that projects sentence embeddings onto the "grammaticality scale"
def mass_mean_score(embeddings):
    """Projects each embedding onto the mass-mean direction, relative to the
    midpoint between the two class means: positive means closer to the
    'grammatical' average, negative means closer to the 'ungrammatical' one."""
    return (embeddings - mass_mean_midpoint) @ mass_mean_direction

# Apply mass_mean_score
scores_mm_good = mass_mean_score(X_blimp_good_norm)
scores_mm_bad = mass_mean_score(X_blimp_bad_norm)

# Calculate accuracy
acc_massmean_blimp = pairwise_accuracy(scores_mm_good, scores_mm_bad)
print(f"PAIRWISE ACCURACY, mass-mean probe on BLiMP: {acc_massmean_blimp:.1%}\n")

# Plot histogram
plot_score_histograms(
    scores_mm_good,
    scores_mm_bad,
    title="Mass-mean probe scores on BLiMP",
    xlabel="Mass-mean projection score (higher = more grammatical-like)",
)


In [ ]:
#@title 5.3: Question: Which method achieved higher pairwise accuracy on BLiMP?
your_answer = "Logistic regression" #@param ["Logistic regression", "Mass-mean probing", "About the same (within 2 points)"]

# Live-graded: compares 4.13's and 5.2's actual accuracies at runtime.
diff_53 = abs(acc_logistic_blimp - acc_massmean_blimp) * 100  # in percentage points

if diff_53 <= 2:
    actual_53 = "About the same (within 2 points)"
elif acc_logistic_blimp > acc_massmean_blimp:
    actual_53 = "Logistic regression"
else:
    actual_53 = "Mass-mean probing"

print(f"Logistic regression accuracy: {acc_logistic_blimp:.1%}")
print(f"Mass-mean probing accuracy:   {acc_massmean_blimp:.1%}")

if your_answer == actual_53:
    print(f"✅ Correct! {actual_53}")
else:
    print(f"❌ Not quite. The correct answer is: {actual_53}")


##**Summary**

If you have completed this notebook, this is what you have accomplished:

**Section 1**: You loaded BERT, looked at its architecture (layers, embedding size, vocabulary), and pulled out contextual embeddings for the word "bat" across three sentences with different meanings. You evaluated whether the similarity between different embeddings of "bat" reflects semantic relatedness.

In the process, you learned about how BERT tokenizes words, dealt with anisotropy by normalizing activations, and compared how similarity patterns between embeddings changed across layers (including the special case of the static embedding layer, which carries no context at all).

**Section 2**: You tried the same kind of analysis on sentences of your own choosing&mdash;just a chance to explore BERT's contextual embeddings.

**Section 3**: You evaluated a psycholinguistic question at a larger scale: does BERT's geometry track the "prototypicality" of double-object vs.
prepositional-object versions of the same sentence (the dative alternation)? You extracted embeddings for minimal DO/PO sentence pairs across all of BERT's layers, using ``[CLS]`` as a token that captures the entire sentence. You then computed the similarities of the two members in each pair, and checked whether those similarities tracked real human preferences from a behavioral dataset.

**Section 4**: You switched from BERT to GPT2-small, and used its last token as a representation of an entire sentence. You used a synthetic corpus of grammatical sentences vs. corrupted versions to train a probing classifier&mdash;a logistic regression model to tell apart these two classes of sentences, using nothing but their embeddings. To check whether the probe identified a representation of grammaticality that was robust and generalizable, you tested it on BLiMP&mdash;a completely different, human-curated benchmark of grammatical violations that the probe had never seen during training.

**Section 5**: You built a second, much simpler probe to create a "grammaticality scale". This scale was defined with the mass-mean approach: just the difference between the average of grammatical sentence embeddings and the average of corrupted sentence embeddings. You created this probe based on the same data the logistic regression probe used, and directly compared the two methods on BLiMP.